In [0]:
-- =============================================================================
-- TRAVEL BOOKING SCD2 MERGE PROJECT - DATA QUALITY SUMMARY
-- =============================================================================
-- This script creates and populates daily data quality summary metrics
-- Purpose: Provides aggregated DQ metrics for monitoring and reporting
-- Business Value: DQ trend analysis, compliance reporting, operational monitoring
-- Dependencies: Requires ops.dq_results table to be populated by DQ notebooks

-- =============================================================================
-- DQ SUMMARY TABLE CREATION
-- =============================================================================
-- Create table for daily data quality summary metrics
-- business_date: Date dimension for time-based DQ analysis
-- dataset: Dataset name (booking_inc, customer_inc, etc.)
-- checks_passed: Number of successful DQ checks
-- checks_failed: Number of failed DQ checks
-- recorded_at: Timestamp when summary was calculated

create table if not exists travel_bookings.ops.dq_daily_summary
(
business_date date,
dataset string,
checks_passed integer,
checks_failed integer,
recorded_at timestamp
) using delta;

-- =============================================================================
-- DQ SUMMARY CALCULATION AND MERGE
-- =============================================================================
-- Calculate daily DQ summary metrics from detailed DQ results
-- Aggregates check results by dataset for high-level monitoring
-- Uses MERGE for idempotent updates (can be run multiple times safely)
-- Parameter binding allows flexible date specification

merge into travel_bookings.ops.dq_daily_summary t
using (
  select  coalesce(try_cast(:arrival_date as date), current_date()) business_date, dataset, sum(case when status='SUCCESS' then 1 else 0 end) checks_passed,
  sum(case when status!='SUCCESS' then 1 else 0 end) checks_failed, current_timestamp() recorded_at from travel_bookings.ops.dq_results 
  where business_date=coalesce(arrival_date, current_date()) 
  group by dataset) s
on s.business_date=t.business_date and s.dataset=t.dataset
WHEN MATCHED THEN UPDATE SET * 
WHEN NOT MATCHED THEN INSERT *;